# Taller 02 — Evidencia de partes faltantes

Este notebook documenta la evidencia para las partes del Taller 02 que requieren código adicional.

**Entorno requerido:** `EMBEDDING_BACKEND=local` (sin VPN, sin Docker, sin clave API)

---
## Parte 0b — El fragmento que se corta a 128 tokens

**Objetivo:** demostrar que el MiniLM trunca silenciosamente cualquier texto que supere su tope de 128 tokens, y que el vector del texto entero es prácticamente idéntico al del texto recortado al tope (coseno ≈ 1), lo que significa que todo lo que no cupió nunca llegó al índice.

In [1]:
# Carga del modelo (embed_notebook_s2 de la tabla semestral)
# fila: embed_notebook_s2 | sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | tope: 128 | verificado: 2026-08-27
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID)
tokenizer = model.tokenizer
MAX_SEQ = model.max_seq_length

print(f"Modelo cargado : {MODEL_ID}")
print(f"Tope del modelo: {MAX_SEQ} tokens  (fila embed_notebook_s2, verificado 2026-08-27)")

c:\Users\User\Documents\ALEXANDER\USFQ\IA_GENERATIVA\Taller2\Lab-02-RAG-VectorSearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3012.68it/s]


Modelo cargado : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Tope del modelo: 128 tokens  (fila embed_notebook_s2, verificado 2026-08-27)


In [2]:
# Texto de prueba de ~900 palabras (generado para la demostración; 
# cualquier texto real de longitud similar produciría el mismo efecto)
texto_900_palabras = """
La inteligencia artificial ha transformado profundamente la forma en que las organizaciones
procesan y analizan grandes volúmenes de información. Desde sus inicios teóricos en la década
de 1950, cuando Alan Turing planteó la pregunta fundamental de si las máquinas podían pensar,
hasta los sistemas de aprendizaje profundo actuales, el camino recorrido ha sido extraordinario.
Los modelos de lenguaje grande, conocidos como LLM por sus siglas en inglés, representan uno
de los avances más significativos de la última década en el campo de la inteligencia artificial.

Estos modelos son entrenados con cantidades masivas de texto proveniente de libros, artículos
académicos, páginas web y otras fuentes escritas, lo que les permite adquirir un conocimiento
amplio sobre una multitud de temas. Sin embargo, este conocimiento tiene una fecha de corte:
el modelo no sabe nada de lo que ocurrió después de la fecha en que fue entrenado. Esto crea
una limitación importante cuando se necesita información actualizada o específica de un dominio
particular que no estaba representado en los datos de entrenamiento.

Para superar esta limitación, surgió la técnica denominada Retrieval-Augmented Generation,
o RAG. La idea fundamental del RAG es sencilla pero poderosa: en lugar de depender únicamente
del conocimiento interno del modelo, el sistema primero busca información relevante en una
base de conocimiento externa y luego usa esa información como contexto para generar la
respuesta. De esta manera, el modelo puede proporcionar respuestas precisas y actualizadas
sin necesidad de ser reentrenado constantemente.

El proceso de RAG consta de varias etapas bien definidas. En primer lugar, se realiza la
ingesta de documentos: los textos del corpus son parseados, divididos en fragmentos más
pequeños y convertidos en vectores numéricos mediante un modelo de embeddings. Estos vectores
capturan el significado semántico del texto de una manera que las computadoras pueden comparar
eficientemente. Los vectores se almacenan en una base de datos vectorial como Qdrant, Pinecone
o Weaviate, que permite realizar búsquedas por similitud de manera muy rápida.

Cuando el usuario formula una pregunta, esa pregunta también se convierte en un vector usando
el mismo modelo de embeddings. Luego, el sistema busca en la base de datos vectorial los
fragmentos cuyo vector sea más similar al de la pregunta. Esta similitud se mide típicamente
con el coseno del ángulo entre los vectores: dos vectores idénticos tienen coseno 1, dos
vectores ortogonales tienen coseno 0. Los fragmentos recuperados se pasan al modelo de
lenguaje como contexto adicional, y el modelo genera una respuesta basada en ese contexto.

La fragmentación del corpus es un paso crítico que tiene un impacto enorme en la calidad
del sistema RAG. Si los fragmentos son demasiado pequeños, pueden no contener suficiente
contexto para responder preguntas complejas. Si son demasiado grandes, pueden superar el
límite de tokens del modelo de embeddings, lo que resulta en truncado silencioso y pérdida
de información. La fragmentación en tokens del mismo tokenizador del modelo es la única
manera de garantizar que los fragmentos no superan el tope máximo de secuencia.

El solapamiento entre fragmentos consecutivos es otra consideración importante. Un solapamiento
de alrededor del veinte por ciento del tamaño del fragmento ayuda a evitar que información
relevante quede dividida entre dos fragmentos sin que ninguno de los dos la contenga completa.
Por ejemplo, si una respuesta importante está al final de un fragmento y al principio del
siguiente, sin solapamiento el sistema podría recuperar solo la mitad de la información
necesaria para responder correctamente a la pregunta del usuario.

La evaluación de un sistema RAG requiere construir lo que se conoce como un golden set: un
conjunto de preguntas con sus respuestas esperadas y sus documentos fuente, anotado manualmente
por expertos en el dominio. Este golden set permite medir métricas como el Hit Rate, que indica
en qué fracción de las preguntas el sistema recuperó al menos un fragmento del documento
correcto, y el MRR, que es la media de los recíprocos de la posición del primer fragmento
correcto en la lista de resultados recuperados.

Una característica fundamental de un sistema RAG bien diseñado es la capacidad de abstención:
cuando la pregunta del usuario no puede responderse con la información disponible en el corpus,
el sistema debe reconocerlo y decirlo explícitamente en lugar de inventar una respuesta. Esto
se logra incluyendo en el prompt una instrucción explícita que autoriza al modelo a admitir
que no tiene suficiente información, y una frase fija de abstención que el evaluador puede
detectar de manera programática para calcular las tasas de abstención correcta e indebida.
"""

palabras = texto_900_palabras.split()
print(f"Palabras en el texto: {len(palabras)}")

Palabras en el texto: 744


In [3]:
# 1) Contar tokens del texto con el tokenizador del modelo
tokens_ids = tokenizer(texto_900_palabras, add_special_tokens=False, truncation=False)["input_ids"]
n_tokens = len(tokens_ids)

print(f"Tokens del texto (~900 palabras) : {n_tokens}")
print(f"Tope del modelo (max_seq_length) : {MAX_SEQ}")
print(f"Tokens que NO llegan al índice   : {max(0, n_tokens - (MAX_SEQ - 2))}")
print()
print("→ El texto tiene significativamente más tokens que el tope del modelo.")
print(f"→ El MiniLM vectoriza solo los primeros {MAX_SEQ - 2} tokens y descarta el resto en silencio.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (988 > 128). Running this sequence through the model will result in indexing errors


Tokens del texto (~900 palabras) : 988
Tope del modelo (max_seq_length) : 128
Tokens que NO llegan al índice   : 862

→ El texto tiene significativamente más tokens que el tope del modelo.
→ El MiniLM vectoriza solo los primeros 126 tokens y descarta el resto en silencio.


In [4]:
# 2) Comparar el vector del texto entero vs. el texto recortado al tope
import numpy as np
from numpy.linalg import norm

# El modelo trunca internamente al vectorizar el texto entero
vec_entero = model.encode(texto_900_palabras)

# Recortamos manualmente al tope (MAX_SEQ - 2 para dejar espacio a tokens especiales)
ids_recortados = tokens_ids[: MAX_SEQ - 2]
texto_recortado = tokenizer.decode(ids_recortados)
vec_recortado = model.encode(texto_recortado)

# Coseno entre ambos vectores
coseno = np.dot(vec_entero, vec_recortado) / (norm(vec_entero) * norm(vec_recortado))

print(f"Coseno(texto_entero, texto_recortado_a_{MAX_SEQ}_tokens) = {coseno:.6f}")
print()
print("Interpretación:")
print(f"  coseno ≈ 1.0 confirma que encode(texto_entero) == encode(texto[:tope])")
print(f"  → Todo lo que pasó del token {MAX_SEQ - 2} fue ignorado por el modelo.")
print(f"  → Con chunk_tokens=900 en la Parte 0b, cada fragmento se indexa truncado a {MAX_SEQ - 2} tokens.")

Coseno(texto_entero, texto_recortado_a_128_tokens) = 1.000000

Interpretación:
  coseno ≈ 1.0 confirma que encode(texto_entero) == encode(texto[:tope])
  → Todo lo que pasó del token 126 fue ignorado por el modelo.
  → Con chunk_tokens=900 en la Parte 0b, cada fragmento se indexa truncado a 126 tokens.


In [5]:
# 3) Justificación del chunk_tokens para la Parte 1 (con bge-m3 en la H200)
BGE_M3_TOPE = 8192  # fila embed_local_multilingue, verificado 2026-08-27
CHUNK_ELEGIDO = 512
OVERLAP_ELEGIDO = CHUNK_ELEGIDO // 5  # 20% de solapamiento

print("=== Parámetros de fragmentación para la Parte 1 ===")
print(f"  Modelo de embeddings : bge-m3 (fila embed_local_multilingue, verificado 2026-08-27)")
print(f"  Tope del modelo      : {BGE_M3_TOPE} tokens")
print(f"  chunk_tokens elegido : {CHUNK_ELEGIDO} tokens")
print(f"  overlap_tokens       : {OVERLAP_ELEGIDO} tokens ({100*OVERLAP_ELEGIDO//CHUNK_ELEGIDO}% del fragmento)")
print()
print("Justificación:")
print(f"  Aunque bge-m3 admite hasta {BGE_M3_TOPE} tokens, un fragmento de ese tamaño")
print( "  equivale a ~6 000 palabras (un capítulo completo), lo que diluye el señal")
print( "  semántico y dificulta que el retriever encuentre la respuesta específica.")
print(f"  512 tokens (~380 palabras) cubre una idea o sección completa del documento,")
print( "  es el tamaño de las notas del curso (§4.2), y es lo suficientemente pequeño")
print( "  para que el coseno sea discriminativo entre fragmentos.")

=== Parámetros de fragmentación para la Parte 1 ===
  Modelo de embeddings : bge-m3 (fila embed_local_multilingue, verificado 2026-08-27)
  Tope del modelo      : 8192 tokens
  chunk_tokens elegido : 512 tokens
  overlap_tokens       : 102 tokens (19% del fragmento)

Justificación:
  Aunque bge-m3 admite hasta 8192 tokens, un fragmento de ese tamaño
  equivale a ~6 000 palabras (un capítulo completo), lo que diluye el señal
  semántico y dificulta que el retriever encuentre la respuesta específica.
  512 tokens (~380 palabras) cubre una idea o sección completa del documento,
  es el tamaño de las notas del curso (§4.2), y es lo suficientemente pequeño
  para que el coseno sea discriminativo entre fragmentos.
